In [1]:
"""
Scraper de mychip.es a traves de la seva API interna (JSON)
==============================================================

CONTEXT — CANVI IMPORTANT respecte a versions anteriors
------------------------------------------------------------
Les versions anteriors d'aquest script feien servir Playwright per simular
un navegador i clicar targetes, perque vam pensar que calia executar el
JavaScript de la pagina per veure les dades. Aixo va donar molts problemes
(Jupyter, Windows, publicitat que no deixava mai la xarxa "en repos",
targetes que no navegaven enlloc...).

Investigant amb les eines de xarxa del navegador hem trobat que la web
carrega TOTES les dades a traves d'una API JSON publica i neta:

    https://sfztfth1r1.execute-api.eu-central-1.amazonaws.com/prod/events
    https://sfztfth1r1.execute-api.eu-central-1.amazonaws.com/prod/registrations
    https://sfztfth1r1.execute-api.eu-central-1.amazonaws.com/prod/registrations/stats

Aixo vol dir que ja NO cal cap navegador ni Playwright: nomes fem servir
la llibreria `requests`, molt mes simple, rapida i fiable. Adeu als
problemes de Jupyter/Windows/clics que no funcionaven.

QUE FA AQUEST SCRIPT
---------------------
1. Recorre /events (POST, amb paginacio offset/limit) filtrant per curses
   finalitzades ("finished_official"/"finished_provisional" = "past").
2. Cada cursa ("event") te una o mes "modalities" (per exemple "7k" i
   "7k + Camiseta" — ATENCIO: de vegades dues modalitats son la MATEIXA
   distancia amb una opcio extra (com una samarreta), no dues curses
   diferents. Si per la teva analisi vols ajuntar-les, hauras d'agrupar
   per (nom de cursa + distancia) despres — aquest script desa cada
   modalitat per separat, tal com les dona l'API).
3. Per cada modalitat, demana:
     - /registrations/stats -> comptadors (registrats, en meta, retirats...)
     - /registrations       -> classificacio completa, corredor per corredor
       (nom, cognoms, genere, club, categoria, dorsal, temps, posicions...)
4. Ho desa tot en 2 CSV:
     - curses.csv: una fila per cada (cursa, modalitat), amb resum
     - classificacions.csv: una fila per cada corredor de cada modalitat
       (el "podi" es pot obtenir despres filtrant pos_gender <= 3)

COM EXECUTAR-HO
-----------------
    pip install requests
    python mychip_api_scraper.py

(funciona igual en Jupyter, en una cel·la: nomes cal cridar main() —
no cal "await" ni "run_async", ja no fem servir cap navegador ni asyncio)

CONFIGURACIO RAPIDA
---------------------
Ajusta les constants de la seccio CONFIG. Ve preparat amb MAX_EVENTS = 5
per fer una PROVA PETITA abans de llançar-ho tot (~2000 curses).
"""

from pathlib import Path
import csv
import json
import os
import time

import requests

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
API_BASE = "https://sfztfth1r1.execute-api.eu-central-1.amazonaws.com/prod"

# Alguns API Gateway/WAF exigeixen que sembli que la peticio ve del navegador
# real. Si reps un error 403 en provar-ho, aquesta capçalera sol ser la causa
# mes habitual — ja la incluim per si de cas.
HEADERS = {
    "Content-Type": "application/json",
    "Origin": "https://www.mychip.es",
    "Referer": "https://www.mychip.es/",
    "User-Agent": "Mozilla/5.0 (compatible; investigacio-academica-tfm)",
}

STATUS_PAST = ["finished_official", "finished_provisional"]

MAX_EVENTS = None        # posa None per no limitar (~2000 curses, trigara una estona)
EVENTS_PAGE_SIZE = 30    # marge de seguretat (el limit real sembla ser ~35-40)
REGISTRATIONS_PAGE_SIZE = 300  # marge de seguretat (el limit real sembla >=400)

SLEEP_BETWEEN_REQUESTS = 1.0  # segons; per no bombardejar l'API
                               # (pujat de 0.2 a 1.0: si l'API comença a
                               # respondre 502 de manera repetida i persistent
                               # (no nomes puntual), sol ser perque esta
                               # limitant/protegint-se d'un ritme de peticions
                               # massa alt — anar mes a poc a poc sol ajudar
                               # a que es recuperi)

MAX_RETRIES = 8            # reintents per a errors transitoris (502/503/504, talls de xarxa...)
RETRY_BACKOFF_SECONDS = 3  # temps d'espera abans de cada reintent (creix i es capa a RETRY_BACKOFF_MAX)
RETRY_BACKOFF_MAX = 30     # espera maxima entre reintents, en segons

SAVE_EVERY_N_EVENTS = 25  # desa un CSV parcial cada N curses, per no perdre feina

OUTPUT_DIR = Path("../../data/raw/mychip")
EVENTS_CSV = os.path.join(OUTPUT_DIR, "DF_MYCHIP_SUCIO.csv")
CLASSIF_CSV = os.path.join(OUTPUT_DIR, "mychip_classificacions.csv")
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "mychip_checkpoint.json")
BLOCKED_EVENTS_FILE = os.path.join(OUTPUT_DIR, "events_bloquejats.json")
# Offsets (posicions dins del llistat d'/events) que ja sabem, d'execucions
# anteriors, que el backend no pot servir de cap manera (502 persistent,
# no puntual). Es desa a disc perque cada vegada que tornis a executar
# main() no calgui repetir tot el proces lent de reintents+subdivisio per
# tornar a descobrir el mateix punt trencat — ja el saltem directament.

# Pausar i reprendre: si atures l'execucio (Ctrl+C, interrompre el kernel...)
# i despres tornes a cridar main(), per defecte CONTINUA on ho havies deixat
# en lloc de tornar a començar — llegeix el checkpoint i els CSV existents
# a OUTPUT_DIR i nomes processa les curses que encara falten.
# Si vols forçar un començament de zero, esborra la carpeta OUTPUT_DIR
# (o posa RESUME = False una vegada).
RESUME = True


# ---------------------------------------------------------------------------
# CRIDES A L'API
# ---------------------------------------------------------------------------
def _post(path, payload):
    """Fa la petició amb reintents automatics per a errors TRANSITORIS
    (502/503/504 del servidor, talls de xarxa breus...). Un 502 "Bad
    Gateway" sol ser un ensopegament puntual d'AWS, no un problema de la
    peticio en si — reintentar-ho uns segons despres sol funcionar.

    Si el MATEIX 502 es repeteix sempre al mateix punt per mes vegades
    que ho reintentis (fins i tot tornant a executar main() de nou),
    normalment NO es un simple ensopegament puntual sinó que l'API esta
    limitant/protegint-se perque hem fet moltes peticions seguides molt
    rapid (rate limiting) — per aixo ara imprimim el cos de la resposta
    (`resp.text`), que sol incloure un missatge mes concret que el codi
    d'estat sol, per poder-ho diagnosticar."""
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.post(f"{API_BASE}{path}", json=payload, headers=HEADERS, timeout=30)
            if resp.status_code in (502, 503, 504):
                cos = (resp.text or "").strip()[:300]
                detall = f" | cos de la resposta: {cos}" if cos else " | (resposta buida)"
                raise requests.exceptions.HTTPError(
                    f"{resp.status_code} transitori{detall}", response=resp
                )
            resp.raise_for_status()
            return resp.json()
        except (requests.exceptions.HTTPError, requests.exceptions.ConnectionError,
                requests.exceptions.Timeout) as e:
            last_error = e
            # Si es un error HTTP que NO es transitori (p.ex. 400/403/404),
            # no te sentit reintentar-ho — ja tornara a fallar igual.
            status = getattr(getattr(e, "response", None), "status_code", None)
            if status is not None and status not in (502, 503, 504):
                raise
            if attempt < MAX_RETRIES:
                wait = min(RETRY_BACKOFF_SECONDS * attempt, RETRY_BACKOFF_MAX)
                print(f"    (avis: {path} offset={payload.get('offset')} — {e} "
                      f"— reintent {attempt}/{MAX_RETRIES} en {wait}s)")
                time.sleep(wait)
    print(f"    (s'esgoten els {MAX_RETRIES} reintents per {path} "
          f"amb payload {payload})")
    raise last_error


def get_events_page(offset, limit=EVENTS_PAGE_SIZE):
    return _post("/events", {
        "offset": offset,
        "limit": limit,
        "order": ["-date_start", "numeric_id"],
        "status": STATUS_PAST,
    })


def get_registrations_page(modality_id, offset=0, limit=REGISTRATIONS_PAGE_SIZE):
    return _post("/registrations", {
        "offset": offset,
        "limit": limit,
        "modality": modality_id,
    })


def get_registration_stats(modality_id):
    return _post("/registrations/stats", {"modality": modality_id})


def _carregar_offsets_bloquejats():
    if not os.path.exists(BLOCKED_EVENTS_FILE):
        return set()
    try:
        with open(BLOCKED_EVENTS_FILE, encoding="utf-8") as f:
            return set(json.load(f))
    except (json.JSONDecodeError, OSError):
        return set()


def _registrar_offset_bloquejat(offset):
    """Desa a disc que aquest offset concret no es pot obtenir de cap
    manera, per no haver de tornar a descobrir-ho (amb tots els reintents
    que aixo comporta) la propera vegada que executis main()."""
    bloquejats = _carregar_offsets_bloquejats()
    if offset in bloquejats:
        return
    bloquejats.add(offset)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(BLOCKED_EVENTS_FILE, "w", encoding="utf-8") as f:
        json.dump(sorted(bloquejats), f)


def _fetch_events_range_resilient(offset, limit, bloquejats=None):
    """Obte events al rang [offset, offset+limit), amb dues proteccions:

    1. Si ja sabem (d'una execucio anterior) que algun offset concret dins
       d'aquest rang es inservible, ens estalviem tornar a provar tot el
       rang sencer (que sabem que tornaria a fallar) i anem directes a
       demanar nomes els trossos bons al voltant seu.
    2. Si un rang falla de manera PERSISTENT sense que ja el coneguem, el
       subdividim en trossos cada cop mes petits per aillar quin registre
       concret fa petar el backend (per exemple, un event amb dades
       malformades que nomes falla per a ell). Quan el trobem, es desa a
       `events_bloquejats.json` (via el punt 1 de dalt, les properes
       vegades) i es salta, en lloc de bloquejar tot l'scraping aqui.

    Torna (files, calia_recuperacio) — `calia_recuperacio` es True si hi ha
    hagut algun error/salt pel cami, per poder distingir "pagina buida
    perque ja no queden mes events" de "pagina buida perque tot ha fallat"."""
    if bloquejats is None:
        bloquejats = _carregar_offsets_bloquejats()

    if limit == 1:
        if offset in bloquejats:
            return [], True
        try:
            return get_events_page(offset, 1), False
        except Exception:
            print(f"    AVIS: no s'ha pogut obtenir l'event a l'offset "
                  f"{offset} de cap manera — es salta definitivament (no "
                  f"bloqueja la resta de l'scraping).")
            _registrar_offset_bloquejat(offset)
            return [], True

    coneguts_al_rang = sorted(o for o in bloquejats if offset <= o < offset + limit)
    if coneguts_al_rang:
        resultats = []
        cursor = offset
        for bad in coneguts_al_rang:
            if bad > cursor:
                bons, _ = _fetch_events_range_resilient(cursor, bad - cursor, bloquejats)
                resultats.extend(bons)
            cursor = bad + 1
        final = offset + limit
        if cursor < final:
            bons, _ = _fetch_events_range_resilient(cursor, final - cursor, bloquejats)
            resultats.extend(bons)
        return resultats, True

    try:
        return get_events_page(offset, limit), False
    except Exception as e:
        meitat = max(1, limit // 2)
        print(f"    (offset={offset} limit={limit} falla de manera "
              f"persistent ({e}) — ho subdivideixo en offset={offset} "
              f"limit={meitat} i offset={offset + meitat} limit={limit - meitat} "
              f"per aillar el problema)")
        primera, _ = _fetch_events_range_resilient(offset, meitat, bloquejats)
        segona, _ = _fetch_events_range_resilient(offset + meitat, limit - meitat, bloquejats)
        return primera + segona, True


def iter_all_events(max_events=None):
    """Genera tots els events (curses) passades, paginant automaticament.

    Fa servir `_fetch_events_range_resilient` per a cada pagina: si una
    pagina falla de manera persistent, s'aillen i se salten nomes els
    registres concrets problematics (i es recorden per no haver-ho de
    tornar a descobrir en properes execucions), en lloc de bloquejar tot
    l'scraping aqui. Com que en aquest cas no podem confiar en "pagina
    buida = ja no queden mes events" (podria ser buida nomes perque ha
    fallat tot el tros), nomes aturem la paginacio quan una pagina surt
    buida SENSE que hagi calgut cap recuperacio.
    """
    offset = 0
    count = 0
    while True:
        page, calia_recuperar = _fetch_events_range_resilient(offset, EVENTS_PAGE_SIZE)
        if not page and not calia_recuperar:
            break

        for ev in page:
            yield ev
            count += 1
            if max_events and count >= max_events:
                return
        offset += EVENTS_PAGE_SIZE
        time.sleep(SLEEP_BETWEEN_REQUESTS)


def iter_all_registrations(modality_id):
    """Genera tots els corredors d'una modalitat, paginant automaticament."""
    offset = 0
    while True:
        page = get_registrations_page(modality_id, offset, REGISTRATIONS_PAGE_SIZE)
        if not page:
            break
        for reg in page:
            yield reg
        if len(page) < REGISTRATIONS_PAGE_SIZE:
            break
        offset += REGISTRATIONS_PAGE_SIZE
        time.sleep(SLEEP_BETWEEN_REQUESTS)


# ---------------------------------------------------------------------------
# TRANSFORMACIO A FILES DE TAULA (facil de provar sense xarxa)
# ---------------------------------------------------------------------------
def _date(d):
    """Extreu la data d'un camp tipus {'$date': '2026-08-01T00:00:00Z'}."""
    if not d:
        return None
    return d.get("$date") if isinstance(d, dict) else d


def gender_breakdown(registrations, predicate):
    """Compta, per genere, quants corredors compleixen `predicate(reg)`.

    Ho calculem nosaltres mateixos a partir de la llista completa de
    corredors (que ja descarreguem per fer la classificacio), en lloc de
    demanar-ho a l'API — aixi no cal esbrinar si /registrations/stats
    accepta cap parametre de genere, i podem aplicar el mateix truc a
    qualsevol altre comptador (en_meta, retirats, etc.) si mes endavant
    en necessites mes desglossats."""
    counts = {}
    for reg in registrations:
        if not predicate(reg):
            continue
        gender = reg.get("gender") or "desconegut"
        counts[gender] = counts.get(gender, 0) + 1
    return counts


def _inscrit(reg):
    return True


def _took_start(reg):
    status = (reg.get("result") or {}).get("status")
    return bool(status) and status != "not_started"


def _finished(reg):
    return (reg.get("result") or {}).get("status") == "finished"


def _not_started(reg):
    return (reg.get("result") or {}).get("status") == "not_started"


def _retired(reg):
    return (reg.get("result") or {}).get("status") == "retired"


# Cada entrada: (nom_columna_base, funcio_que_decideix_si_compta)
# S'utilitza tant per calcular els totals com el desglossament per genere,
# aixi nomes cal afegir una linia aqui si algun dia vols un altre recompte.
_METRIQUES = [
    ("inscrits", _inscrit),
    ("sortida_donada", _took_start),
    ("en_meta", _finished),
    ("no_presentats", _not_started),
    ("retirats", _retired),
]


def event_row(event, modality, stats, registrations=None):
    loc = event.get("location") or {}
    coords = (loc.get("coordinates") or {}).get("coordinates") or [None, None]
    status = stats.get("status", {}) if stats else {}

    registrations = registrations or []

    row = {
        "event_numeric_id": event.get("numeric_id"),
        "event_slug": event.get("slug"),
        "event_url": f"https://www.mychip.es/e/{event.get('slug')}" if event.get("slug") else None,
        "nom_cursa": event.get("name"),
        "data": _date(event.get("date_start")),
        "ciutat": loc.get("city"),
        "comarca_regio": loc.get("region"),
        "provincia_estat": loc.get("state"),
        "pais": loc.get("country"),
        "coord_1": coords[0],
        "coord_2": coords[1],
        "modalitat_id": modality.get("id"),
        "modalitat_nom": modality.get("name"),
        "distancia_m": modality.get("distance"),
        "esport": modality.get("sport"),
        "estat_cursa": event.get("status"),
        # Totals (ambdos sexes), de /registrations/stats quan hi son
        # disponibles i, si no, calculats nosaltres mateixos a partir
        # de la llista de corredors (fallback per si stats ha fallat).
        "total_inscrits": stats.get("total") if stats else len(registrations),
        "en_meta": status.get("finished") if status else None,
        "sortida_donada": status.get("took_start") if status else None,
        "no_presentats": status.get("not_started") if status else None,
        "retirats": status.get("retired") if status else None,
    }

    # Desglossament per genere de cada metrica (inscrits, sortida_donada,
    # en_meta/arribats, no_presentats, retirats), calculat directament a
    # partir de la llista de corredors — aixi ja no cal cap script a part
    # per obtenir una fila per cursa+modalitat amb els participants
    # desglossats per sexe: ja surt tot directament del scraping.
    generes_trobats = set()
    per_metrica = {}
    for nom_metrica, predicat in _METRIQUES:
        counts = gender_breakdown(registrations, predicat)
        per_metrica[nom_metrica] = counts
        generes_trobats.update(counts.keys())

    for nom_metrica, counts in per_metrica.items():
        row[f"{nom_metrica}_h"] = counts.get("male", 0)
        row[f"{nom_metrica}_d"] = counts.get("female", 0)
        # Si apareix algun altre valor de genere (per exemple "other" o
        # "desconegut"), l'afegim igualment amb una columna propia en lloc
        # de perdre'l, en lloc de forçar-lo dins de h/d.
        for gender in generes_trobats:
            if gender not in ("male", "female"):
                row[f"{nom_metrica}_{gender}"] = counts.get(gender, 0)

    return row


def registration_row(event, modality, reg):
    result = reg.get("result") or {}
    return {
        "event_numeric_id": event.get("numeric_id"),
        "nom_cursa": event.get("name"),
        "modalitat_id": modality.get("id"),
        "modalitat_nom": modality.get("name"),
        "dorsal": reg.get("bib_number"),
        "nom": reg.get("name"),
        "cognoms": reg.get("surname"),
        "genere": reg.get("gender"),
        "club": reg.get("club") or None,
        "categoria": reg.get("category"),
        "cancel_lat": reg.get("cancelled"),
        "estat_resultat": result.get("status"),
        "temps_total_segons": result.get("total_time"),
        "pos_general": result.get("pos_general"),
        "pos_genere": result.get("pos_gender"),
        "pos_categoria": result.get("pos_category"),
        "pos_club": result.get("pos_club"),
    }


def slim_modality(modality):
    """Extreu nomes els camps utils d'una modalitat (evitem arrossegar dades
    redundants — cada modalitat porta l'event sencer aninat a dins, etc.)."""
    return {
        "id": modality["_id"]["$oid"] if isinstance(modality.get("_id"), dict) else modality.get("_id"),
        "numeric_id": modality.get("numeric_id"),
        "name": modality.get("name"),
        "short_name": modality.get("short_name"),
        "distance": modality.get("distance"),
        "sport": modality.get("sport"),
        "type": modality.get("type"),
        "status": modality.get("status"),
    }


# ---------------------------------------------------------------------------
# ORQUESTRACIO
# ---------------------------------------------------------------------------
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    already_done = _load_checkpoint() if RESUME else 0
    event_rows = _load_csv(EVENTS_CSV) if RESUME else []
    registration_rows = _load_csv(CLASSIF_CSV) if RESUME else []

    # Xarxa de seguretat: el checkpoint no hauria MAI de ser mes petit que
    # el nombre de curses diferents que ja hi ha realment desades a
    # curses.csv. Si aixo passa (per exemple perque s'ha executat sense
    # voler una versio antiga de l'script, amb un MAX_EVENTS de prova mes
    # petit que el progres real — exactament el bug que vam trobar), ho
    # detectem i ho corregim automaticament aqui, en lloc de dependre que
    # algu ho repari a ma cada vegada.
    if RESUME and event_rows:
        curses_al_csv = len({row.get("event_numeric_id") for row in event_rows})
        if curses_al_csv > already_done:
            print(f"AVIS: el checkpoint deia {already_done} curses fetes, pero "
                  f"{EVENTS_CSV} ja en te {curses_al_csv} de diferents — "
                  f"s'utilitza {curses_al_csv} per no repetir ni duplicar feina "
                  f"que ja estava feta.")
            already_done = curses_al_csv
            _save_checkpoint(already_done)

    if already_done:
        print(f"Represenent des de la cursa {already_done + 1} "
              f"(ja hi havia {already_done} curses processades — "
              f"{len(event_rows)} files a {EVENTS_CSV}, "
              f"{len(registration_rows)} a {CLASSIF_CSV}).")
        print("(Si vols començar de zero, esborra la carpeta "
              f"'{OUTPUT_DIR}' o posa RESUME = False.)\n")

    # Si passa qualsevol cosa (un error persistent de l'API, una
    # interrupcio manual amb Ctrl+C/Kernel Interrupt...), el bloc
    # "finally" de mes avall desa igualment tot el que s'hagi recollit
    # fins aquell moment — no es perd la feina feta, i la propera vegada
    # que executis main() continuara des d'aqui.
    try:
        _run(event_rows, registration_rows, already_done)
    except KeyboardInterrupt:
        print("\nInterromput manualment. Desant el que hi ha fins ara...")
    except Exception as e:
        # Aixo nomes pot passar per un error que ha esgotat tots els
        # reintents (per exemple, si l'API es queda "penjada" en 502
        # durant mes temps del que aguanten els reintents). No es perd
        # res: tot el que ja s'havia processat es desa igualment mes
        # avall. Si torna a fallar al MATEIX punt cada vegada que
        # tornis a cridar main(), molt probablement no es un error
        # puntual sinó que l'API esta limitant les peticions (rate
        # limiting) perque n'hem fet moltes seguides — en aquest cas,
        # esperar uns minuts (o mitja hora) abans de tornar-ho a provar
        # sol ajudar mes que reintentar-ho immediatament.
        print(f"\nS'ha aturat per un error persistent: {e}")
        print("No es perd res del que ja s'havia processat (es desa igualment "
              "mes avall). Torna-ho a provar mes tard executant main() de nou "
              "— continuara des d'on ho ha deixat.")
    finally:
        _write_csv(EVENTS_CSV, event_rows)
        _write_csv(CLASSIF_CSV, registration_rows)
        print(f"\nCurses/modalitats: {len(event_rows)} files -> {EVENTS_CSV}")
        print(f"Corredors: {len(registration_rows)} files -> {CLASSIF_CSV}")

    return event_rows, registration_rows


def _run(event_rows, registration_rows, already_done=0):
    # IMPORTANT: MAX_EVENTS limita el nombre TOTAL de curses que es demanen
    # des del principi del llistat (offset 0), NO "quantes mes a partir d'on
    # ho vas deixar". Si MAX_EVENTS es mes petit o igual que already_done
    # (per exemple, tornes a executar amb el MAX_EVENTS=5 de prova despres
    # d'haver arribat molt mes lluny), el bucle nomes recorre curses ja fetes
    # i no hi ha res a processar — ho avisem explicitament en lloc de
    # deixar que sembli que "no fa res".
    if MAX_EVENTS and MAX_EVENTS <= already_done:
        print(f"AVIS: MAX_EVENTS ({MAX_EVENTS}) es mes petit o igual que les "
              f"curses ja processades ({already_done}) — no hi ha res nou a "
              f"fer amb aquest limit. Puja MAX_EVENTS (o posa'l a None per no "
              f"limitar) i torna a executar main() per continuar.")
        return already_done

    idx = already_done  # per si el bucle no itera cap vegada (p.ex. ja esta tot fet)
    for idx, event in enumerate(iter_all_events(MAX_EVENTS), 1):
        if idx <= already_done:
            # Ja processada en una execucio anterior (veure RESUME):
            # nomes cal tornar a demanar el llistat d'events (rapid), nomes
            # ens estalviem les crides cares de stats/registrations.
            continue

        try:
            _process_event(event, idx, event_rows, registration_rows)
        except Exception as e:
            # Una cursa que falli (per exemple, si esgota els reintents
            # d'un error persistent) no ha de tirar per terra tota la
            # resta — la saltem i seguim amb la seguent.
            print(f"  ERROR processant la cursa {event.get('name')!r}: {e} — la salto")

        if idx % SAVE_EVERY_N_EVENTS == 0:
            _write_csv(EVENTS_CSV, event_rows)
            _write_csv(CLASSIF_CSV, registration_rows)
            _save_checkpoint(idx)
            print(f"  (progres desat: {idx} curses processades fins ara)")

    # Checkpoint final, per si l'ultim tram no queia just en un multiple
    # de SAVE_EVERY_N_EVENTS. Fem servir max(...) perque MAI baixi el
    # checkpoint per sota d'on ja estava (aquest era exactament el bug:
    # si el bucle nomes iterava curses ja fetes, `idx` quedava per sota
    # de already_done i aixo "retrocedia" el checkpoint).
    _save_checkpoint(max(idx, already_done))
    return max(idx, already_done)


def _process_event(event, idx, event_rows, registration_rows):
        modalities = [slim_modality(m) for m in (event.get("modalities") or [])]
        print(f"[{idx}] {event.get('name')} — {len(modalities)} modalitat(s)")

        for modality in modalities:
            mod_id = modality.get("id")
            if not mod_id:
                continue

            try:
                stats = get_registration_stats(mod_id)
            except Exception as e:
                print(f"    ERROR stats modalitat {modality.get('name')}: {e}")
                stats = {}

            raw_regs = []
            try:
                raw_regs = list(iter_all_registrations(mod_id))
                print(f"    {modality.get('name')}: {len(raw_regs)} corredors")
            except Exception as e:
                print(f"    ERROR registrations modalitat {modality.get('name')}: {e}")

            # Passem la llista de corredors a event_row perque calculi el
            # desglossament per genere (p.ex. sortida_donada_h/d) a partir
            # de les dades reals, sense haver de demanar-ho a l'API a part.
            event_rows.append(event_row(event, modality, stats, raw_regs))

            for reg in raw_regs:
                registration_rows.append(registration_row(event, modality, reg))

            time.sleep(SLEEP_BETWEEN_REQUESTS)


def _write_csv(path, rows):
    if not rows:
        return

    # Xarxa de seguretat: aquesta funcio NOMES hauria d'escriure un fitxer
    # amb MES o IGUAL nombre de files que el que ja hi ha al disc — mai
    # menys. Aixo protegeix contra qualsevol bug (present o futur, per
    # exemple si per algun motiu `event_rows`/`registration_rows` es carrega
    # buit o incomplet a l'inici de main(), en un directori equivocat,
    # etc.) que faci que sobreescrivim un CSV gran amb un de mes petit i
    # perdem feina feta. Si detectem que passaria aixo, NO sobreescrivim:
    # desem a un fitxer ".NOMES_LECTURA_revisa.csv" a part i avisem, perque
    # ho puguis revisar tu abans que es perdi res.
    if RESUME and os.path.exists(path):
        existent = _load_csv(path)
        if len(existent) > len(rows):
            path_revisio = path + ".NOMES_LECTURA_revisa.csv"
            fieldnames = sorted({k for row in rows for k in row.keys()})
            with open(path_revisio, "w", newline="", encoding="utf-8-sig") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()
                writer.writerows(rows)
            print(f"AVIS IMPORTANT: {path} ja te {len(existent)} files i "
                  f"anavem a escriure'n nomes {len(rows)} — aixo sembla un "
                  f"error (es perdria feina feta), aixi que NO he tocat "
                  f"{path}. He desat les {len(rows)} files noves a "
                  f"'{path_revisio}' perque ho revisis tu abans de decidir "
                  f"que fer.")
            return

    fieldnames = sorted({k for row in rows for k in row.keys()})
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def _load_csv(path):
    """Torna a carregar un CSV ja desat com a llista de dicts (per continuar
    una execucio interrompuda sense perdre el que ja s'havia desat)."""
    if not os.path.exists(path):
        return []
    with open(path, newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))


def _load_checkpoint():
    if not os.path.exists(CHECKPOINT_FILE):
        return 0
    with open(CHECKPOINT_FILE, encoding="utf-8") as f:
        return json.load(f).get("completed_events", 0)


def _save_checkpoint(completed_events):
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump({"completed_events": completed_events}, f)


if __name__ == "__main__":
    main()

Represenent des de la cursa 1987 (ja hi havia 1986 curses processades — 5150 files a ../../data/raw/mychip/DF_MYCHIP_SUCIO.csv, 91408 a ../../data/raw/mychip/mychip_classificacions.csv).
(Si vols començar de zero, esborra la carpeta '../../data/raw/mychip' o posa RESUME = False.)

    (avis: /events offset=30 — 502 transitori | cos de la resposta: {"message": "Internal server error"} — reintent 1/8 en 3s)
    (avis: /events offset=30 — 502 transitori | cos de la resposta: {"message": "Internal server error"} — reintent 2/8 en 6s)
    (avis: /events offset=30 — 502 transitori | cos de la resposta: {"message": "Internal server error"} — reintent 3/8 en 9s)
    (avis: /events offset=30 — 502 transitori | cos de la resposta: {"message": "Internal server error"} — reintent 4/8 en 12s)
    (avis: /events offset=30 — 502 transitori | cos de la resposta: {"message": "Internal server error"} — reintent 5/8 en 15s)
    (avis: /events offset=30 — 502 transitori | cos de la resposta: {"message": "

In [2]:
import pandas as pd

curses = pd.read_csv("../../data/raw/mychip/DF_MYCHIP_SUCIO.csv", parse_dates=["data"])

print(curses.shape)
curses.sample(10)


(5153, 36)


,ciutat,comarca_regio,coord_1,coord_2,data,distancia_m,en_meta,en_meta_d,en_meta_h,en_meta_other,...,provincia_estat,retirats,retirats_d,retirats_h,retirats_other,sortida_donada,sortida_donada_d,sortida_donada_h,sortida_donada_other,total_inscrits
2692,Elche,El Bajo Vinalopó,38.265331,-0.698839,2019-12-29 00:00:00+00:00,12000.0,NaN,0,0,NaN,...,Comunidad Valenciana,NaN,0,0,NaN,0,0,0,NaN,0
3664,la Pobla Llarga,La Ribera Alta,39.085280,-0.476533,2017-10-01 00:00:00+00:00,107000.0,758.0,37,721,NaN,...,Comunidad Valenciana,3.0,0,3,NaN,761,37,724,NaN,898
1924,Béjar,NaN,40.386581,-5.764962,2022-07-02 00:00:00+00:00,60000.0,33.0,5,28,NaN,...,Castilla y León,6.0,0,6,NaN,39,5,34,NaN,45
4304,Torrevieja,NaN,37.977542,-0.682845,2016-02-28 00:00:00+00:00,10000.0,386.0,107,279,NaN,...,Comunidad Valenciana,NaN,0,0,NaN,386,107,279,NaN,444
4095,Burgos,NaN,42.343926,-3.696977,2016-10-09 00:00:00+00:00,21000.0,618.0,78,540,NaN,...,Castilla y León,15.0,2,13,NaN,633,80,553,NaN,706
879,la Vall d'Uixó,la Plana Baixa,39.823768,-0.245609,2025-03-09 00:00:00+00:00,10300.0,102.0,34,68,NaN,...,Comunitat Valenciana,NaN,0,0,NaN,102,34,68,NaN,112
2686,Toledo,NaN,39.856068,-4.023957,2019-12-31 00:00:00+00:00,400.0,195.0,60,135,NaN,...,Castilla-La Mancha,NaN,0,0,NaN,195,60,135,NaN,225
2449,Cuatretonda,La Vall d'Albaida,38.945699,-0.401626,2021-05-02 00:00:00+00:00,25000.0,NaN,0,0,NaN,...,Comunidad Valenciana,NaN,0,0,NaN,0,0,0,NaN,0
2289,Alcoy,La Hoya de Alcoy,38.698228,-0.474777,2021-11-13 00:00:00+00:00,NaN,NaN,0,0,NaN,...,Comunidad Valenciana,NaN,0,0,NaN,0,0,0,NaN,0
2807,Sellent,La Ribera Alta,39.031798,-0.587475,2019-10-20 00:00:00+00:00,21000.0,NaN,0,0,NaN,...,Comunidad Valenciana,NaN,0,0,NaN,0,0,0,NaN,0
